# T4 — DVC: versionamento de dados

**Objetivo:** versionar dados como versionamos código, sem colocar os dados no Git, e ligar cada modelo
à versão exata dos dados que o treinou.

| Peça | Onde fica | Vai para o Git? |
|---|---|---|
| `bike.csv` (o dado) | sua pasta + cache `.dvc/cache` + **remote** | ❌ |
| `bike.csv.dvc` (ponteiro: hash md5 + tamanho) | ao lado do dado | ✅ |
| `.dvc/config` (onde fica o remote) | `.dvc/` | ✅ |

> 💡 O DVC e o remote (`../dvc-storage`) foram configurados na T1. Lá, o pipeline só **anotava o hash** do
> `bike.csv` no `dvc.lock`. Agora o próprio dado passa a ser versionado.
>
> Os comandos que mudam o Git e o DVC vêm **comentados** (com `#`), para não rodarem sem querer.
> Quando chegar a hora, tire o `#` (selecione as linhas e use **Ctrl + /**) e rode a célula.

In [1]:
import os, sys
from pathlib import Path

# Trabalha a partir da raiz do repositório (a pasta que contém o params.yaml)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "params.yaml").exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

---
# Parte 1 · Juntos

In [2]:
%%bash
git status --short
git log --oneline | head -5
dvc remote list

 M .gitignore
 M aula01/t0_ambiente.ipynb
 M aula01/t1_pipeline_dvc.ipynb
?? aula01/t2_mlflow_tracking.ipynb
?? aula01/t3_model_registry.ipynb
?? aula01/t4_dvc_dados.ipynb
?? data/
e7ac2ba HGB: max_leaf_nodes 15
c608b21 Pipeline DVC: + evaluate e plot
a58e448 Inicializa DVC com remote local
83223dd Add files via upload
3a7bfe1 Add files via upload
local   /workspaces/dvc-storage (default)


## 1. Versionar os dados v1 (2011)

São sempre os mesmos 4 passos. **Guarde esta receita: você vai repeti-la na Parte 2.**

| Passo | Comando | O que faz |
|---|---|---|
| 1 | `dvc add data/raw/bike.csv` | calcula o md5, guarda no cache e cria o ponteiro `.dvc` |
| 2 | `git add data/raw/bike.csv.dvc` + `git commit` | o **ponteiro** vai para o Git |
| 3 | `git tag dados-v1` | um nome fácil para esta versão |
| 4 | `dvc push` | o **dado** vai para o remote |

In [3]:
%%bash
dvc add data/raw/bike.csv
cat data/raw/bike.csv.dvc

⠋ Checking graph



To track the changes with git, run:

	git add data/raw/bike.csv.dvc

To enable auto staging, run:

	dvc config core.autostage true
outs:
- md5: fc7d423afbce5699a61ac1a8acc533b8
  size: 562742
  hash: md5
  path: bike.csv


O `.gitignore` do template já ignora CSVs dentro de `data/`, por isso o DVC não precisou criar um novo.

In [4]:
%%bash
git add data/raw/bike.csv.dvc
git commit -m "Dados v1: 2011"
git tag dados-v1
dvc push

[main db2fdb8] Dados v1: 2011
 1 file changed, 5 insertions(+)
 create mode 100644 data/raw/bike.csv.dvc
1 file pushed


## 2. Desastre: apagaram o dado **e** o cache

Se o dado está no remote, basta um `dvc pull`.

In [6]:
%%bash
#rm -rf data/raw/bike.csv .dvc/cache
dvc pull
wc -l data/raw/bike.csv

A       data/raw/bike.csv
9 files fetched and 1 file added
8646 data/raw/bike.csv


## 3. Como voltar no tempo (você vai usar na Parte 2)

Para trazer uma versão antiga são **2 comandos**: o Git traz o **ponteiro** antigo e o
`dvc checkout` traz o **dado** correspondente.

```bash
git checkout <tag> -- data/raw/bike.csv.dvc   # ponteiro da versão <tag>
dvc checkout                                  # dado que corresponde ao ponteiro
```
Para voltar à versão atual, troque `<tag>` por `HEAD`.

---
# Parte 2 · Sozinhos

Chegaram dados novos: o **1º semestre de 2012**. Versione-os com a mesma receita da Parte 1.
Repare no `dvc status`: ele mostra que o dado mudou **e** que o estágio `prepare` do pipeline, que depende dele, ficou desatualizado.

In [7]:
%%bash
python src/get_data.py --until 2012-06-30
dvc status

data/raw/bike.csv: 13,003 linhas | 2011-01-01 a 2012-06-30
prepare:
	changed deps:
		modified:           data/raw/bike.csv
train:
	changed deps:
		params.yaml:
			modified:           train
data/raw/bike.csv.dvc:
	changed outs:
		modified:           data/raw/bike.csv


### 1. Versione os dados v2 (receita do passo 1)

In [9]:
%%bash
# TODO: os 4 passos da receita, agora com a tag dados-v2 e a mensagem "Dados v2: até jun/2012"
dvc add data/raw/bike.csv
git add data/raw/bike.csv.dvc
git commit -m "Dados v2: até jun/2012"
git tag dados-v2
dvc push
git log --oneline --decorate | head -5

⠋ Checking graph



To track the changes with git, run:

	git add data/raw/bike.csv.dvc

To enable auto staging, run:

	dvc config core.autostage true
[main e92a8e9] Dados v2: até jun/2012
 1 file changed, 2 insertions(+), 2 deletions(-)


fatal: tag 'dados-v2' already exists


1 file pushed
e92a8e9 (HEAD -> main) Dados v2: até jun/2012
db2fdb8 (tag: dados-v2, tag: dados-v1) Dados v1: 2011
e7ac2ba (origin/main, origin/HEAD) HGB: max_leaf_nodes 15
c608b21 Pipeline DVC: + evaluate e plot
a58e448 Inicializa DVC com remote local


### 2. Viagem no tempo: volte para a v1, conte as linhas e retorne para a v2 (passo 3)

In [10]:
%%bash
# TODO: traga a versão dados-v1 (2 comandos)
git checkout dados-v1 -- data/raw/bike.csv.dvc
dvc checkout
wc -l data/raw/bike.csv      # v1: 8.646 linhas (8.645 + cabeçalho)

M       data/raw/bike.csv
8646 data/raw/bike.csv


In [11]:
%%bash
# TODO: volte para a versão atual (HEAD)
git checkout HEAD -- data/raw/bike.csv.dvc
dvc checkout
wc -l data/raw/bike.csv      # v2: 13.004 linhas

M       data/raw/bike.csv
13004 data/raw/bike.csv


### 3. Retreine o pipeline com os dados v2 (o `dvc repro` da T1)

Como o dado mudou, o pipeline inteiro roda de novo. O treino gera um novo run no MLflow e uma nova versão no Registry.

In [14]:
%%bash
# TODO: rode o pipeline e mostre as métricas (dvc metrics show)
dvc repro
dvc metrics show
dvc metrics diff dados-v1 

'data/raw/bike.csv.dvc' didn't change, skipping
Stage 'prepare' didn't change, skipping
Stage 'split' didn't change, skipping
Stage 'train' didn't change, skipping
Stage 'evaluate' didn't change, skipping
Stage 'plot' didn't change, skipping
Data and pipelines are up to date.
Path          mae      r2       rmse
metrics.json  35.5516  0.94818  50.53559
Path          Metric    dados-v1    workspace    Change
metrics.json  mae       34.9424     35.5516      0.6092
metrics.json  r2        0.72296     0.94818      0.22522
metrics.json  rmse      54.74162    50.53559     -4.20604


### 4. Guarde o recibo e publique no GitHub: o código, os ponteiros e as tags (os dados não vão)

In [15]:
%%bash
# TODO: commit de dvc.lock, metrics.json e reports/previsto_vs_real.png; depois dvc push, git push e git push --tags
git add dvc.lock metrics.json reports/previsto_vs_real.png
git commit -m "Retreino com dados v2"
dvc push
git push
git push --tags

On branch main
Your branch is up to date with 'origin/main'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   .gitignore
	modified:   aula01/t0_ambiente.ipynb
	modified:   aula01/t1_pipeline_dvc.ipynb

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	aula01/t2_mlflow_tracking.ipynb
	aula01/t3_model_registry.ipynb
	aula01/t4_dvc_dados.ipynb
	data/.gitignore

no changes added to commit (use "git add" and/or "git commit -a")


Everything is up to date.


Everything up-to-date
Everything up-to-date
